# V4 Training Diagnostic
## Why V4 AUC collapsed from 0.740 → 0.576

This notebook investigates the root cause of V4's poor AUC compared to V3.

**Key findings:**
- `is_home` stored as bool (dropped by trainer) — V3 had it as float64
- H2H features 87% zeros in V4 vs fully populated in V3
- 239 columns with 50+ zero-variance noise vs V3's clean 146 columns

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load datasets
# Copy v3_original from Hetzner: scp root@5.161.239.229:/home/sportsuite/sport-suite/nba/features/datasets/xl_training_POINTS_2023_2025_batched.csv /tmp/v3_original_POINTS.csv
v3_df = pd.read_csv('/tmp/v3_original_POINTS.csv')
v4_df = pd.read_csv('nba/features/datasets/xl_v2_matchup_training_POINTS_2023_2025.csv')

print(f'V3 original: {v3_df.shape}')
print(f'V4 new:      {v4_df.shape}')

## 1. is_home — The Silent Drop

In [ ]:
print('=== is_home ===')
print(f'V3 dtype: {v3_df["is_home"].dtype}, values: {v3_df["is_home"].unique()}')
print(f'V4 dtype: {v4_df["is_home"].dtype}, values: {v4_df["is_home"].unique()}')
print()
print('V3: float64 (0.0/1.0) — included in training')
print('V4: bool (True/False) — DROPPED by select_dtypes(include="number")')
print()
print(f'V3 is_home importance in model: #7 regressor (285), critical feature')
print(f'V4 is_home importance: NOT IN TOP 20 (silently dropped)')

## 2. H2H Features — Population Rate

In [ ]:
h2h_cols_v3 = [c for c in v3_df.columns if c.startswith('h2h_')]
h2h_cols_v4 = [c for c in v4_df.columns if c.startswith('h2h_')]

print(f'V3 H2H columns: {len(h2h_cols_v3)}')
print(f'V4 H2H columns: {len(h2h_cols_v4)}')
print()

# Compare population rates for key H2H features
compare = []
for c in ['h2h_avg_points', 'h2h_std_points', 'h2h_games', 'h2h_home_avg_points', 
          'h2h_away_avg_points', 'h2h_L3_points', 'h2h_L5_points', 'h2h_decayed_avg_points']:
    v3_pct = (v3_df[c] != 0).mean() * 100 if c in v3_df.columns else 0
    v4_pct = (v4_df[c] != 0).mean() * 100 if c in v4_df.columns else 0
    compare.append({'feature': c, 'V3 populated %': v3_pct, 'V4 populated %': v4_pct, 'delta': v4_pct - v3_pct})

cdf = pd.DataFrame(compare)
print(cdf.to_string(index=False))
print()
print(f'V3 avg h2h_games: {v3_df["h2h_games"].mean():.1f}')
print(f'V4 avg h2h_games: {v4_df["h2h_games"].mean():.1f}')

In [ ]:
# Visualize H2H population rates
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# V3 H2H distribution
axes[0].hist(v3_df['h2h_games'], bins=20, alpha=0.7, color='green', edgecolor='black')
axes[0].set_title(f'V3: h2h_games distribution\nmean={v3_df["h2h_games"].mean():.1f}')
axes[0].set_xlabel('H2H Games')

# V4 H2H distribution
axes[1].hist(v4_df['h2h_games'], bins=20, alpha=0.7, color='red', edgecolor='black')
axes[1].set_title(f'V4: h2h_games distribution\nmean={v4_df["h2h_games"].mean():.1f}')
axes[1].set_xlabel('H2H Games')

plt.tight_layout()
plt.savefig('notebooks/h2h_comparison.png', dpi=150)
plt.show()

## 3. Feature Count & Zero-Variance

In [ ]:
# V3 zero-variance features
v3_numeric = v3_df.select_dtypes(include='number')
v3_zero_var = [c for c in v3_numeric.columns if v3_numeric[c].std() == 0]

# V4 zero-variance features
v4_numeric = v4_df.select_dtypes(include='number')
v4_zero_var = [c for c in v4_numeric.columns if v4_numeric[c].std() == 0]

print(f'V3: {len(v3_numeric.columns)} numeric features, {len(v3_zero_var)} zero-variance')
print(f'V4: {len(v4_numeric.columns)} numeric features, {len(v4_zero_var)} zero-variance')
print(f'\nV3 usable features: {len(v3_numeric.columns) - len(v3_zero_var)}')
print(f'V4 usable features: {len(v4_numeric.columns) - len(v4_zero_var)}')
print(f'\nV4 zero-variance features:')
for c in sorted(v4_zero_var):
    print(f'  {c} = {v4_df[c].iloc[0]}')

## 4. Feature Importance Comparison (V3 vs V4)

In [ ]:
# Load both models
v3_reg = pickle.load(open('nba/models/saved_xl/points_v3_regressor.pkl', 'rb'))
v3_feat = pickle.load(open('nba/models/saved_xl/points_v3_features.pkl', 'rb'))
v4_reg = pickle.load(open('nba/models/saved_xl/points_v4_regressor.pkl', 'rb'))
v4_feat = pickle.load(open('nba/models/saved_xl/points_v4_features.pkl', 'rb'))

# V3 top 20
v3_imp = pd.DataFrame({'feature': v3_feat, 'importance': v3_reg.feature_importances_})
v3_imp = v3_imp.sort_values('importance', ascending=False).head(20)

# V4 top 20
v4_imp = pd.DataFrame({'feature': v4_feat, 'importance': v4_reg.feature_importances_})
v4_imp = v4_imp.sort_values('importance', ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].barh(v3_imp['feature'][::-1], v3_imp['importance'][::-1], color='green', alpha=0.7)
axes[0].set_title('V3 Regressor — Top 20 Features\n(AUC: 0.740)')
axes[0].set_xlabel('Importance')

axes[1].barh(v4_imp['feature'][::-1], v4_imp['importance'][::-1], color='orange', alpha=0.7)
axes[1].set_title('V4 Regressor — Top 20 Features\n(AUC: 0.576)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('notebooks/feature_importance_v3_vs_v4.png', dpi=150)
plt.show()

## 5. V4 New Features — Are They Adding Signal?

In [ ]:
# Correlations of new V4 features with target
v3_feature_set = set(v3_feat)
new_features = [f for f in v4_feat if f not in v3_feature_set]

print(f'V4-only features: {len(new_features)}')
print()

# Compute correlation with label
corrs = []
for f in new_features:
    if f in v4_df.columns and v4_df[f].std() > 0:
        c = v4_df[f].corr(v4_df['label'])
        pop = (v4_df[f] != 0).mean()
        corrs.append({'feature': f, 'corr_with_target': c, 'populated_pct': pop})

corr_df = pd.DataFrame(corrs).sort_values('corr_with_target', ascending=False)
print('New V4 features by correlation with target (label):')
print(corr_df.to_string(index=False))

## 6. Date Range Comparison

In [ ]:
print('=== Date Ranges ===')
print(f'V3: {v3_df["game_date"].min()} to {v3_df["game_date"].max()} ({len(v3_df)} rows)')
print(f'V4: {v4_df["game_date"].min()} to {v4_df["game_date"].max()} ({len(v4_df)} rows)')
print()

# Props per month
v3_dates = pd.to_datetime(v3_df['game_date'])
v4_dates = pd.to_datetime(v4_df['game_date'])

print('V3 props by month:')
print(v3_dates.dt.to_period('M').value_counts().sort_index().to_string())
print()
print('V4 props by month:')
print(v4_dates.dt.to_period('M').value_counts().sort_index().to_string())

## 7. Root Cause Summary

| Issue | V3 (0.740 AUC) | V4 (0.576 AUC) | Impact |
|-------|----------------|----------------|--------|
| `is_home` | float64 (0/1) | bool (dropped) | **CRITICAL** — #7 feature lost |
| H2H population | ~70% non-zero | 13% non-zero | **CRITICAL** — top 4 features gutted |
| Zero-variance | ~5 columns | 50+ columns | **HIGH** — dilutes signal |
| Feature count | 146 clean | 239 noisy | **HIGH** — curse of dimensionality |
| Date range | Oct 2023 - Dec 2025 | Oct 2023 - Mar 2026 | Positive (more data) |
| New BP features | N/A | 15 features | Moderate positive signal |

### Fix Plan:
1. Convert `is_home` to int in dataset builder (not trainer)
2. Recompute matchup_history for ALL seasons before building dataset
3. Drop zero-variance features before training
4. Consider training with V3 features + cherry-picked V4 features